# AF2·07 — FAPE Loss & Backbone Generation

**Mechanism of the day:** turn the reader (IPA) into a *writer*. Each residue's updated
representation predicts a **frame update** — a small rotation and translation that nudges
the residue — and iterating these updates from a blank start folds a backbone. Then the
loss that makes it trainable: **FAPE**, the frame-aligned point error.

Rung 06 gave us IPA: every residue gathers geometric context from the others, invariantly.
But IPA only produces a *representation*; it does not move anything. The structure module
closes the loop:

```
start: all frames at identity / origin (the "black hole")
repeat:  s <- s + IPA(s, z, frames)          # read geometry (rung 06)
         (rot, trans) <- head(s)             # each residue proposes how to move
         frames <- update(frames, rot, trans)# apply the move
```

After a handful of iterations the frames spell out a structure. The remaining question is
how to *score* a predicted structure against the truth so we can train — and here a naive
choice (say, RMSD after alignment) is awkward and not obviously the right invariance.
AlphaFold uses **FAPE**:

> express every atom in *every* residue's local frame, for both the prediction and the
> truth, and average the (clamped) distance between them.

Because it compares atoms *within local frames*, FAPE is invariant to global pose — the
same symmetry IPA respects — and because it aligns under *every* frame, not one global
alignment, it penalises local geometric errors everywhere. You will build the frame
update and FAPE, wire a mini structure module, and **fold toy proteins from their
distance map**, generalising to held-out structures.

**How to use this notebook:** implement the reps, make the checkpoints pass. Solutions at
the bottom. Trains in ~30s on a laptop CPU.

In [ ]:
import math, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0); rng = np.random.default_rng(0)
plt.rcParams['axes.spines.top'] = False; plt.rcParams['axes.spines.right'] = False
BLUE, GREEN, INK = '#2a78d6', '#008300', '#52514e'
L = 16; c = 48; cz = 24; NITER = 8

# ---- toy 3D proteins + their distance map (given) ----
def make_structure():
    pos = [np.zeros(3)]; d = rng.normal(0, 1, 3); d /= np.linalg.norm(d)
    for _ in range(L - 1):
        d = d + rng.normal(0, 0.5, 3); d /= np.linalg.norm(d)
        pos.append(pos[-1] + d - 0.03 * pos[-1])
    X = np.array(pos); X -= X.mean(0); return X.astype(np.float32)
DIST_BINS = np.linspace(1.5, 9.0, 15)
def disto_bins(X):
    D = np.sqrt(((X[:, None] - X[None]) ** 2).sum(-1)); return np.digitize(D, DIST_BINS).astype(np.int64)
def build(n):
    Xs = [make_structure() for _ in range(n)]
    return torch.tensor(np.array(Xs)), torch.tensor(np.array([disto_bins(X) for X in Xs]))
Xtr, Dtr = build(200); Xte, Dte = build(40)

def hat(w):
    O = torch.zeros(*w.shape[:-1], 3, 3)
    O[..., 0, 1]=-w[..., 2]; O[..., 0, 2]=w[..., 1]; O[..., 1, 0]=w[..., 2]
    O[..., 1, 2]=-w[..., 0]; O[..., 2, 0]=-w[..., 1]; O[..., 2, 1]=w[..., 0]
    return O
def so3_exp(w):
    '''rotation vector -> rotation matrix (from rung 1.6).'''
    th = w.norm(dim=-1, keepdim=True).clamp_min(1e-8); K = hat(w / th); th = th[..., None]
    return torch.eye(3).expand_as(K) + torch.sin(th) * K + (1 - torch.cos(th)) * (K @ K)
def true_frames(X):
    '''consistent backbone frames from the CA trace (Gram-Schmidt on neighbours).'''
    R = torch.zeros(L, 3, 3)
    for i in range(L):
        a = X[min(i + 1, L - 1)] - X[i]; b = X[max(i - 1, 0)] - X[i]
        e1 = a / (a.norm() + 1e-8); e2 = b - (e1 @ b) * e1; e2 = e2 / (e2.norm() + 1e-8)
        R[i] = torch.stack([e1, e2, torch.cross(e1, e2, dim=-1)], -1)
    return R
Rtr = [true_frames(Xtr[i]) for i in range(len(Xtr))]
Rte = [true_frames(Xte[i]) for i in range(len(Xte))]
print('%d train + %d test toy proteins | folding target: their Ca backbone' % (len(Xtr), len(Xte)))

## Part 1 — FAPE: the frame-aligned point error

FAPE compares a predicted structure `(t_pred, R_pred)` with the true `(t_true, R_true)`.
For **every** residue frame `i` and **every** atom `j`, it expresses atom `j` in frame
`i`'s local coordinates (both for prediction and truth) and measures how far apart they
land:

$$
\mathrm{FAPE} = \operatorname*{mean}_{(i,j)} \operatorname{clamp}\!\Big(\big\lVert \text{to-local}(T_i^{p}, x_j^{p}) - \text{to-local}(T_i^{t}, x_j^{t})\big\rVert,\ d_{\text{clamp}}\Big)
$$

We use `Cα`-only atoms (so `x_j = t_j`), which keeps the code short while capturing the
idea. Two things to appreciate:

- **Invariance for free.** Both structures are viewed in *local* frames, so a global
  rotation/translation of either changes nothing — the same symmetry as IPA.
- **Aligned everywhere.** Using every frame `i` (not one global superposition) means a
  local kink is penalised in the frames near it, wherever it is. The clamp keeps large
  early errors from dominating the gradient.

### Rep 1 — `fape(tp, Rp, tt, Rt, clamp=10.0)`
`tp, tt` are `[L,3]` predicted/true `Cα` positions; `Rp, Rt` are `[L,3,3]` frames. For
each frame `i`, put every `Cα_j` into local coordinates via `R_iᵀ(t_j − t_i)`, take the
distance between predicted and true, clamp, and average over all `(i, j)`.

In [ ]:
def fape(tp, Rp, tt, Rt, clamp=10.0):
    '''Frame-aligned point error (Ca-only), averaged over all (frame i, atom j) pairs.'''
    # YOUR CODE HERE
    # hint: local coords of atom j in frame i:  einsum('lji,lkj->lki', R, t[None] - t[:, None])
    #       (that applies R_i^T to (t_j - t_i)); do it for pred and true;
    #       err = (loc_p - loc_t).norm(dim=-1).clamp(max=clamp); return err.mean()
    raise NotImplementedError

# --- checkpoint ---
X = Xtr[0]; Rt = Rtr[0]
assert fape(X, Rt, X, Rt).item() < 1e-5, 'a structure vs itself has zero FAPE'
# FAPE is invariant to a global move of the prediction
Rg = so3_exp(torch.randn(1, 3))[0]; tg = torch.randn(3)
Xm = X @ Rg.T + tg; Rm = torch.einsum('ij,njk->nik', Rg, Rt)
assert abs(fape(Xm, Rm, X, Rt).item()) < 1e-4, 'FAPE must be invariant to a global pose change'
# a genuinely wrong structure has large FAPE
assert fape(torch.tensor(make_structure()), true_frames(torch.tensor(make_structure())), X, Rt).item() > 0.5
print('FAPE ok — zero against itself, pose-invariant, large for a wrong fold ✓')

## Part 2 — the frame update

Each residue's representation predicts six numbers: a rotation vector (3) and a
translation (3), both in the residue's **local** frame. We apply them by composing onto
the current frame:

$$
\begin{aligned} R_{\text{new}} &= R\,\exp(\text{rot vec}) \\ t_{\text{new}} &= t + R_{\text{new}}\cdot\text{trans} \end{aligned}
$$

The rotation vector is scaled down so each step is a gentle nudge — the module refines the
structure over several iterations rather than teleporting it.

### Rep 2 — `frame_update(R, t, rot, trans)`
`R` `[L,3,3]`, `t` `[L,3]`, and predicted `rot, trans` each `[L,3]` (local). Return the
updated `(R_new, t_new)` per the rule above. Use the given `so3_exp`.

In [ ]:
def frame_update(R, t, rot, trans):
    '''Compose a predicted local (rotation, translation) update onto the current frames.'''
    # YOUR CODE HERE
    # hint: R_new = R @ so3_exp(rot); t_new = t + torch.einsum('lij,lj->li', R_new, trans)
    raise NotImplementedError

# --- checkpoint ---
R0 = torch.eye(3).expand(L, 3, 3).contiguous(); t0 = torch.zeros(L, 3)
Rn, tn = frame_update(R0, t0, torch.zeros(L, 3), torch.zeros(L, 3))
assert torch.allclose(Rn, R0) and torch.allclose(tn, t0), 'a zero update changes nothing'
Rn, tn = frame_update(R0, t0, torch.zeros(L, 3), torch.ones(L, 3))
assert torch.allclose(tn, torch.ones(L, 3), atol=1e-5), 'from identity, a unit local step is a unit global step'
Rn, _ = frame_update(R0, t0, torch.tensor([[0., 0., 1.0]]).expand(L, 3).contiguous(), torch.zeros(L, 3))
assert torch.allclose(Rn @ Rn.transpose(-1, -2), torch.eye(3).expand(L, 3, 3), atol=1e-5), 'stays a rotation'
print('frame update ok — composes a local nudge onto each residue frame ✓')

## Part 3 — the mini structure module (given), and the iteration

IPA is the compact version from rung 06. The structure module below embeds the input
distance map into the pair representation `z`, starts every frame at the **black hole**
(identity rotation, origin), and runs `NITER` iterations of *read (IPA) → propose
(update head) → move (your `frame_update`)*. It records the frames at every step so we can
apply FAPE across the whole trajectory (this "deep supervision" is what AlphaFold does —
it stabilises the early iterations).

In [ ]:
h, dd, Np, Npv = 4, 8, 4, 6
class IPA(nn.Module):
    def __init__(s):
        super().__init__()
        s.qs=nn.Linear(c,h*dd); s.ks=nn.Linear(c,h*dd); s.vs=nn.Linear(c,h*dd)
        s.qp=nn.Linear(c,h*Np*3); s.kp=nn.Linear(c,h*Np*3); s.vp=nn.Linear(c,h*Npv*3)
        s.bz=nn.Linear(cz,h); s.gamma=nn.Parameter(torch.zeros(h))
        s.out=nn.Linear(h*dd+h*cz+h*Npv*3+h*Npv, c)
    def forward(s, x, z, R, t):
        qs=s.qs(x).view(L,h,dd); ks=s.ks(x).view(L,h,dd); vs=s.vs(x).view(L,h,dd)
        g=lambda lp: torch.einsum('lij,lhpj->lhpi', R, lp) + t[:,None,None,:]
        Qg=g(s.qp(x).view(L,h,Np,3)); Kg=g(s.kp(x).view(L,h,Np,3)); Vg=g(s.vp(x).view(L,h,Npv,3))
        scal=torch.einsum('ihd,jhd->ijh', qs, ks)/math.sqrt(dd)
        d2=((Qg[:,None]-Kg[None])**2).sum(-1).sum(-1)
        a=F.softmax(scal + s.bz(z) - 0.5*F.softplus(s.gamma)[None,None]*d2, 1)
        o_s=torch.einsum('ijh,jhd->ihd', a, vs).reshape(L,h*dd)
        o_z=torch.einsum('ijh,ijz->ihz', a, z).reshape(L,h*cz)
        og=torch.einsum('ijh,jhpx->ihpx', a, Vg); ol=torch.einsum('lji,lhpj->lhpi', R, og-t[:,None,None,:])
        return s.out(torch.cat([o_s, o_z, ol.reshape(L,h*Npv*3), ol.norm(dim=-1).reshape(L,h*Npv)], -1))

class StructureModule(nn.Module):
    def __init__(s):
        super().__init__()
        s.zemb=nn.Embedding(16, cz); s.sinit=nn.Parameter(torch.zeros(c)); s.pos=nn.Embedding(L, c)
        s.ipa=IPA(); s.ln=nn.LayerNorm(c); s.head=nn.Linear(c, 6)     # 3 rot + 3 trans
    def forward(s, dbins):
        z = s.zemb(dbins)
        x = s.sinit[None].expand(L, c) + s.pos(torch.arange(L))
        R = torch.eye(3).expand(L, 3, 3).contiguous(); t = torch.zeros(L, 3)   # black-hole init
        traj = []
        for _ in range(NITER):
            x = s.ln(x + s.ipa(x, z, R, t))
            u = s.head(x); R, t = frame_update(R, t, u[:, :3] * 0.3, u[:, 3:])
            traj.append((R, t))
        return R, t, traj
print('mini structure module ready — %d refinement iterations from the black-hole init' % NITER)

## Part 4 — fold it

Train the module to fold each protein from its distance map, with FAPE summed over the
trajectory. Then, on **held-out** proteins, fold from the distogram alone and measure
FAPE and the aligned `Cα` RMSD against a random-fold baseline.

In [ ]:
def kabsch_rmsd(P, Q):
    '''RMSD after optimal rigid superposition.'''
    Pc = P - P.mean(0); Qc = Q - Q.mean(0); H = Pc.T @ Qc
    U, S, Vt = torch.linalg.svd(H); dsign = torch.sign(torch.det(Vt.T @ U.T))
    Rr = Vt.T @ torch.diag(torch.tensor([1., 1., dsign])) @ U.T
    return ((Rr @ Pc.T).T - Qc).norm(dim=-1).pow(2).mean().sqrt()

net = StructureModule(); opt = torch.optim.AdamW(net.parameters(), lr=3e-3)
t0 = time.time(); hist = []
for step in range(1400):
    i = int(rng.integers(len(Xtr)))
    R, t, traj = net(Dtr[i])
    loss = sum(fape(tt, RR, Xtr[i], Rtr[i]) for (RR, tt) in traj) / len(traj)   # deep supervision
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 100 == 0: hist.append((step, loss.item()))
print('trained in %.0fs' % (time.time() - t0))

net.eval()
with torch.no_grad():
    fapes = [fape(net(Dte[i])[1], net(Dte[i])[0], Xte[i], Rte[i]).item() for i in range(len(Xte))]
    rmsds = [kabsch_rmsd(net(Dte[i])[1], Xte[i]).item() for i in range(len(Xte))]
bh = fape(torch.zeros(L, 3), torch.eye(3).expand(L, 3, 3), Xte[0], Rte[0]).item()
rand_rmsd = np.mean([kabsch_rmsd(torch.tensor(make_structure()), Xte[i]).item() for i in range(len(Xte))])

### Rep 3 — `fold_quality(rmsds, random_baseline)`
Return the fraction of the random-baseline RMSD that our folds achieve,
`mean(rmsds) / random_baseline` — below `1.0` means we fold better than chance. A tiny
rep, but it forces the honest comparison: is the module actually folding, or just
producing compact blobs that score okay?

In [ ]:
def fold_quality(rmsds, random_baseline):
    '''mean predicted RMSD as a fraction of the random-fold baseline (lower = better).'''
    # YOUR CODE HERE
    raise NotImplementedError

# --- checkpoint ---
q = fold_quality(rmsds, rand_rmsd)
assert np.mean(fapes) < bh - 0.5, 'trained FAPE must beat the black-hole init clearly'
assert q < 0.75, 'folds should be well below the random-baseline RMSD'
print('held-out FAPE %.2f  (black-hole init %.2f)' % (np.mean(fapes), bh))
print('held-out aligned Ca-RMSD %.2f  (random-fold baseline %.2f)  ->  %.0f%% of random'
      % (np.mean(rmsds), rand_rmsd, 100 * q))
print('\nThe module folds recognisably from the distance map alone. (It under-extends the')
print('longest loops — real sub-angstrom accuracy needs the full model, weight-shared')
print('iterations with recycling, and real training. But every mechanism here is AF2.) ✓')

fig, ax = plt.subplots(figsize=(5.4, 3.2))
hh = np.array(hist); ax.plot(hh[:, 0], hh[:, 1], color=BLUE, lw=2)
ax.axhline(bh, color=INK, ls='--', lw=1.5); ax.text(hh[-1, 0], bh, ' black-hole init', va='bottom', ha='right', fontsize=9, color=INK)
ax.set_xlabel('step'); ax.set_ylabel('FAPE (trajectory mean)'); ax.set_title('learning to fold'); ax.grid(alpha=.15)
plt.show()

In [ ]:
# See the folds: predicted (blue) vs true (green), optimally superposed.
def superpose(P, Q):
    Pc = P - P.mean(0); Qc = Q - Q.mean(0); H = Pc.T @ Qc
    U, S, Vt = torch.linalg.svd(H); dsign = torch.sign(torch.det(Vt.T @ U.T))
    Rr = Vt.T @ torch.diag(torch.tensor([1., 1., dsign])) @ U.T
    return (Rr @ Pc.T).T, Qc

fig = plt.figure(figsize=(12, 3.7))
with torch.no_grad():
    for k in range(3):
        _, t, _ = net(Dte[k]); P, Qc = superpose(t, Xte[k])
        ax = fig.add_subplot(1, 3, k + 1, projection='3d')
        ax.plot(*Qc.T.numpy(), '-o', ms=3, color=GREEN, lw=1.5, label='true')
        ax.plot(*P.T.numpy(), '-o', ms=3, color=BLUE, lw=1.5, label='folded')
        ax.set_title('held-out protein %d (RMSD %.2f)' % (k, kabsch_rmsd(t, Xte[k])), fontsize=9)
        ax.set_xticklabels([]); ax.set_yticklabels([]); ax.set_zticklabels([])
        if k == 0: ax.legend(fontsize=8)
plt.tight_layout(); plt.show()
print('Blue tracks green: a backbone folded from noise, driven by IPA and scored by FAPE. ✓')

## Reflection — what just transferred

- **The structure module writes structure by editing frames.** From a black-hole init,
  each iteration reads geometry with IPA, predicts a small local `(rotation, translation)`
  per residue, and composes it onto the frames. A few iterations fold a backbone.
- **FAPE is the right loss** for this world: it views every atom in every local frame, so
  it is invariant to global pose (like IPA) and penalises local errors everywhere. Zero
  against the truth, large for a wrong fold, blind to how the whole thing is oriented.
- **Deep supervision** — applying FAPE at every iteration, not just the last — stabilises
  the refinement, exactly as AlphaFold does.
- **It genuinely folds.** From a distance map alone, on proteins it never saw, the module
  reaches well under the random-fold RMSD. It is not sub-angstrom — that needs scale,
  weight-shared iterations, and recycling — but the machinery is complete and correct.

**Next rung:** `AF2·08 — Recycling & confidence`. Two finishing pieces: **recycling** —
feed the whole prediction back through the network several times to refine it — and the
**confidence heads** (pLDDT, PAE) that let AlphaFold tell you *which parts of its own
prediction to trust*.

---
Scroll down only after you've done the reps.

## Solutions appendix (peek only after trying)

In [ ]:
def fape(tp, Rp, tt, Rt, clamp=10.0):
    loc_p = torch.einsum('lji,lkj->lki', Rp, tp[None] - tp[:, None])   # R_i^T (t_j - t_i), predicted
    loc_t = torch.einsum('lji,lkj->lki', Rt, tt[None] - tt[:, None])   # ... true
    return (loc_p - loc_t).norm(dim=-1).clamp(max=clamp).mean()

def frame_update(R, t, rot, trans):
    R_new = R @ so3_exp(rot)
    t_new = t + torch.einsum('lij,lj->li', R_new, trans)
    return R_new, t_new

def fold_quality(rmsds, random_baseline):
    return float(np.mean(rmsds)) / random_baseline

print('reference solutions loaded — re-run the checkpoint cells above')